In [13]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os

In [14]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
MODEL_NAME = "LSTM Classifier"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"

# Hyperparameters
INPUT_SIZE = 1
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

In [15]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [16]:
# Preprocessing
X = df.drop('Output', axis=1)
y = df['Output']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

print("\nSplitting the dataset into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("\nFeature scaling complete.")
print(f"Scaled training features shape: {X_train_scaled.shape}")
print(f"Scaled testing features shape: {X_test_scaled.shape}")


Features (X) shape: (510530, 13)
Target (y) shape: (510530,)

Splitting the dataset into training and testing sets...
Training features shape: (102106, 13)
Testing features shape: (408424, 13)
Training target shape: (102106,)
Testing target shape: (408424,)

Feature scaling complete.
Scaled training features shape: (102106, 13)
Scaled testing features shape: (408424, 13)


In [17]:
# Prepare data for LSTM: (samples, sequence_length, input_size)
X_train_np = X_train_scaled.values.astype('float32')
X_test_np = X_test_scaled.values.astype('float32')
y_train_np = y_train.values.astype('int64')
y_test_np = y_test.values.astype('int64')

SEQUENCE_LENGTH = X_train_np.shape[1]   # 13 features per sample

X_train_lstm = X_train_np.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
X_test_lstm = X_test_np.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)

# Convert to torch tensors
X_train_tensor = torch.tensor(X_train_lstm)
y_train_tensor = torch.tensor(y_train_np)
X_test_tensor = torch.tensor(X_test_lstm)
y_test_tensor = torch.tensor(y_test_np)

In [18]:
# Create DataLoader for batching
train_data = TensorDataset(X_train_tensor, y_train_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

In [19]:
# Define a basic LSTM model for classification
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [21]:
# initialize the model, loss function, and optimizer
model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [22]:
# Training loop
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

train_time = time.time() - start_time

Epoch 1/100, Loss: 0.6450, Accuracy: 0.7849
Epoch 2/100, Loss: 0.4053, Accuracy: 0.8200
Epoch 3/100, Loss: 0.3431, Accuracy: 0.8404
Epoch 4/100, Loss: 0.3160, Accuracy: 0.8501
Epoch 5/100, Loss: 0.2958, Accuracy: 0.8558
Epoch 6/100, Loss: 0.2821, Accuracy: 0.8612
Epoch 7/100, Loss: 0.2697, Accuracy: 0.8664
Epoch 8/100, Loss: 0.2606, Accuracy: 0.8713
Epoch 9/100, Loss: 0.2490, Accuracy: 0.8765
Epoch 10/100, Loss: 0.2376, Accuracy: 0.8827
Epoch 11/100, Loss: 0.2290, Accuracy: 0.8852
Epoch 12/100, Loss: 0.2222, Accuracy: 0.8877
Epoch 13/100, Loss: 0.2136, Accuracy: 0.8919
Epoch 14/100, Loss: 0.2128, Accuracy: 0.8914
Epoch 15/100, Loss: 0.2072, Accuracy: 0.8936
Epoch 16/100, Loss: 0.2005, Accuracy: 0.8966
Epoch 17/100, Loss: 0.1959, Accuracy: 0.8995
Epoch 18/100, Loss: 0.1893, Accuracy: 0.9021
Epoch 19/100, Loss: 0.1893, Accuracy: 0.9023
Epoch 20/100, Loss: 0.1853, Accuracy: 0.9026
Epoch 21/100, Loss: 0.1827, Accuracy: 0.9045
Epoch 22/100, Loss: 0.1763, Accuracy: 0.9074
Epoch 23/100, Loss:

In [ ]:
# Evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)
        all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print("Accuracy:", accuracy * 100)
print("F1 Score:", f1 * 100)

In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference_torch


_timing = time_inference_torch(model, next(iter(test_loader))[0].to(device))

row = evaluate(
    all_labels,
    all_preds,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="LSTM",
    labels=list(range(NUM_CLASSES)),
    class_names=["benign", "simple", "intermediate", "sophisticated"],
    y_proba=all_probs,
    y_train=y_train,
    train_time_sec=train_time,
    num_epochs=NUM_EPOCHS,
    parameters=list(dict(model.named_parameters()).keys()),
    seed=RANDOM_STATE,
    n_features=X_test_scaled.shape[1],
    n_train=len(y_train),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")